<div class="alert alert-info">
    <b>2.4 Filter Database (Ionic Balance Error)</b><br>
    Dieses Notebook filtert die aktuellste 'Komplette_Datenbank' basierend auf dem Ionenbilanzfehler (IBE), wie in der Main-Ions-Six Analyse definiert.<br>
    -> <b>Ziel:</b> Nur Proben behalten, deren IBE innerhalb von +/- 5% liegt UND deren Temperatur >= 10°C ist<br>
    -> <b>Output:</b> Speichert eine NEUE Version (mit Zeitstempel) der 'Komplette_Datenbank.csv' für den nächsten Pipeline-Schritt (3.1 Preprocessing).
</div>

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import datetime
import shutil

In [2]:
# ----------- Einstellungen ---------------
IBE_THRESHOLD = 5.0  # Prozent (+/-)
TEMP_THRESHOLD = 10.0 # Grad Celsius (Minimum)
REQUIRED_IONS = ['Na', 'Ca', 'Mg', 'Cl', 'HCO3', 'SO4']

# ------------------ Valenz-Wörterbuch zur Normalisierung ----------------------
ION_SPECS = {
    'Na':   {'valence': 1, 'mass': 22.99, 'type': 'cation', 'regex': r'Na_in_([a-zA-Z0-9/]+)'},
    'Ca':   {'valence': 2, 'mass': 40.08, 'type': 'cation', 'regex': r'Ca_in_([a-zA-Z0-9/]+)'},
    'Mg':   {'valence': 2, 'mass': 24.31, 'type': 'cation', 'regex': r'Mg_in_([a-zA-Z0-9/]+)'},
    'Cl':   {'valence': 1, 'mass': 35.45, 'type': 'anion',  'regex': r'Cl_in_([a-zA-Z0-9/]+)'},
    'HCO3': {'valence': 1, 'mass': 61.02, 'type': 'anion',  'regex': r'HCO3_in_([a-zA-Z0-9/]+)'},
    'SO4':  {'valence': 2, 'mass': 96.06, 'type': 'anion',  'regex': r'SO4_in_([a-zA-Z0-9/]+)'},
}

### 1. Laden der aktuellsten Datenbank

In [3]:
base_dir = Path.cwd()

notebooks_root = base_dir.parent
db_archive_path = notebooks_root / "1.1_Data-Acquisition-Wrapper" / "Angepasste_Datenbanken" / "Komplette_Datenbank"

if not db_archive_path.exists():
    raise FileNotFoundError(f"Datenbank-Archivpfad nicht gefunden: {db_archive_path}")

# ------------------------ Finde den neuesten Zeitstempel-Ordner -----------------------------
all_folders = [f for f in db_archive_path.iterdir() if f.is_dir()]
latest_folder = max(all_folders, key=lambda x: x.name)
input_csv = latest_folder / "Komplette_Datenbank.csv"

print(f"Lade aktuellste Datenbank: {input_csv}")
df = pd.read_csv(input_csv, low_memory=False)
print(f"Anzahl geladener Zeilen: {len(df)}")

Lade aktuellste Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Komplette_Datenbank\2025-12-30_11-30-56\Komplette_Datenbank.csv


Anzahl geladener Zeilen: 175099


### 2. Logik zur IBE-Berechnung

In [4]:
def get_conversion_factor(unit, valence, mass):
    """Gibt den Umrechnungsfaktor zu meq/L zurück"""
    unit = unit.lower()
    if unit == 'mmol/l':
        return valence
    elif unit == 'mg/l':
        return valence / mass
    elif unit == 'meq/l':
        return 1.0
    elif unit == 'ppm':
        # ----------------------------- Annahme Dichte ~1kg/L, ppm ~ mg/L ----------------------------------
        return valence / mass
    return 0.0

def calculate_ibe(df):
    """Berechnet den Ladungsbilanzfehler (Charge Balance Error) NUR für die definierten 6 Hauptionen."""
    
    cations_sum = pd.Series(0.0, index=df.index)
    anions_sum = pd.Series(0.0, index=df.index)
    found_cols = []
    
    # ----------------------------------Identifiziere Spalten im Dataframe --------------------------------
    for ion, specs in ION_SPECS.items():
        regex = specs['regex']
        # --------------- Finde passende Spalte -----------------
        matches = [c for c in df.columns if re.search(regex, c)]
        
        if len(matches) > 1:
            print(f"Warnung: Mehrere Spalten für {ion} gefunden: {matches}. Verwende die erste.")
        
        if matches:
            col_name = matches[0]
            found_cols.append(col_name)
            
            # --------------------------- Extrahiere Einheit aus dem Spaltennamen -------------------------
            unit_match = re.search(regex, col_name)
            if unit_match:
                unit = unit_match.group(1)
                factor = get_conversion_factor(unit, specs['valence'], specs['mass'])
                
                # ------------- Konvertiere zu numerisch, setze nicht-numerische als NaN -> 0 -------------
                vals = pd.to_numeric(df[col_name], errors='coerce').fillna(0)
                meq_vals = vals * factor
                
                if specs['type'] == 'cation':
                    cations_sum += meq_vals
                else:
                    anions_sum += meq_vals
    
    total_ions = cations_sum + anions_sum
    
    # -------------------- Berechne IBE (Formel: (cat - an) / (cat + an) * 100) ----------------------
    # Vermeidung von Division durch Null
    ibe = pd.Series(np.nan, index=df.index)
    valid_mask = total_ions > 0.0001
    
    ibe[valid_mask] = ((cations_sum[valid_mask] - anions_sum[valid_mask]) / total_ions[valid_mask]) * 100
    
    return ibe, valid_mask

### 3. Filter anwenden

In [5]:
print("Berechne IBE...")
ibe_values, valid_sum_mask = calculate_ibe(df)

# ------------------ Wende Schwellenwert-Filter an (IBE und Temperatur) -----------------------
temp_vals = pd.to_numeric(df['temperature_in_c'], errors='coerce')
filter_mask = (ibe_values.abs() <= IBE_THRESHOLD) & valid_sum_mask & (temp_vals >= TEMP_THRESHOLD)

df_filtered = df[filter_mask].copy()

removed_count = len(df) - len(df_filtered)
print(f"\nFilter Bericht:")
print(f"Zeilen Original:  {len(df)}")
print(f"Zeilen Gefiltert: {len(df_filtered)}")
print(f"Entfernte Zeilen: {removed_count} ({(removed_count/len(df))*100:.2f}%)")

Berechne IBE...

Filter Bericht:
Zeilen Original:  175099
Zeilen Gefiltert: 8464
Entfernte Zeilen: 166635 (95.17%)


### 4. Gefilterte Datenbank speichern

In [6]:
# -------------------- Erzeuge neuen Zeitstempel-Ordner ---------------------------
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
new_output_folder = db_archive_path / timestamp

if not new_output_folder.exists():
    new_output_folder.mkdir(parents=True)

output_file = new_output_folder / "Komplette_Datenbank.csv"

df_filtered.to_csv(output_file, index=False)
print(f"Gefilterte Datenbank gespeichert unter:\n{output_file}")

# --------------- Überprüfung für den nächsten Pipeline-Schritt -----------------
print(f"Erstellter Ordner: {new_output_folder.name}")

Gefilterte Datenbank gespeichert unter:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Komplette_Datenbank\2025-12-30_11-38-35\Komplette_Datenbank.csv
Erstellter Ordner: 2025-12-30_11-38-35
